In [2]:
import yaml
import torch
import pytorch_lightning as pl
from tqdm.auto import tqdm
import torch.nn as nn
import os
import numpy as np 

In [3]:
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"


In [4]:
config_path = "config/binaural_attn/word_task_half_co_loc_v08_gender_bal_4M_w_no_cue_learned.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 0
config['shuffle_train'] = False
config['hparas']['batch_size'] = config['hparas']['batch_size'] // 4 
config['audio']['rep_kwargs']['rep_on_gpu'] = True
del config['corpus']['cue_free_percentage']



In [5]:
import src.spatial_attn_lightning as binaural_lightning 
module = binaural_lightning.BinauralAttentionModule(config)

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


Using explicit dim specification for demeaning in audio transforms
Using BinauralAuditoryAttentionCNN
v08 True
num_classes={'num_words': 800}
Model performing word task
Conv block order: LN -> Conv -> ReLU
coch_affine: True
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [6]:
train_loader = module.train_dataloader()

Using v06 dataset
/om/scratch/Sun/imgriff/datasets/spatial_audio_pipeline/assets/dataset_binaural_attn/v08
Using gender balanced training 4M set
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
913 files in train concat dataset
len training set = 50215
shuffle train: False


In [7]:
audio_transforms = module.audio_transforms

### Run times based on shuffling in h5:
* True: ~ 10it/s
* False: ~14it/s - note LOTS of osscilation in i/o speed here 

In [8]:
ix = 100
for batch in tqdm(train_loader):
    ix -= 1 
    if ix < 0:
        break
    # pass
    # break

  0%|          | 0/50215 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [8]:
# cue, _, scene, labels = batch

In [9]:
audio_transforms

AudioCompose(
    AudioToTensor()
    BinauralCombineWithRandomDBSNR()
    BinauralRMSNormalizeForegroundAndBackground()
    UnsqueezeAudio()
)

In [10]:
from re import L
import h5py
import torch
import glob
import sys
import os 
# sys.path.append('/om4/group/mcdermott/user/imgriff/projects/End-to-end-ASR-Pytorch')
# import src.audio_transforms as audio_transforms
import pickle
import numpy as np


class BinauralAttentionDataset(torch.utils.data.ConcatDataset):
    # Makes a dataset using pre-paired speech and audioset background sounds
    
    def __init__(self, root, cue_type, task, batch_size=1, skip_negative_elev=False, mode='train', with_cue_free=False, **kwargs):
        """
        Builds the pytorch hdf5 combined dataset from the files found in the
        specified root directory. 
        """
        self.hdf5_glob = '*.hdf5_chunk000' if with_cue_free else 'noise*.hdf5_chunk000' 

        if mode == 'train':
            self.all_hdf5_files = glob.glob(root + '/train/' + self.hdf5_glob)
            # screen dead files 
            self.all_hdf5_files = [fname for fname in self.all_hdf5_files  if os.path.getsize(fname) > 0]
        elif mode == 'val':
            self.all_hdf5_files = glob.glob(root + '/validation/' + self.hdf5_glob) 
        elif mode == 'test':
            self.all_hdf5_files = glob.glob(root + '/test/' + self.hdf5_glob) 

        # read files to skip from a file
        with open(root + '/bad_files.txt', 'r') as f:
            files_to_skip = [line.strip().split('/')[-1] for line in f.readlines()]
        # filter bad files from the dataset
        self.all_hdf5_files = [fname for fname in self.all_hdf5_files if fname.split('/')[-1] not in files_to_skip]

        print(f"{len( self.all_hdf5_files)} files in {mode} concat dataset")
        self.all_hdf5_datasets = [H5Dataset(h5_file, cue_type, task, batch_size, skip_negative_elev) for h5_file in self.all_hdf5_files]

        super().__init__(self.all_hdf5_datasets)

    def class_map(self):
        """
        Loads the mapping between the word IDX and human readable word map. 
        """
        class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
        return class_map


class H5Dataset(torch.utils.data.Dataset):
    def __init__(self, path, cue_type, task, batch_size, skip_negative_elev=False):
        """
        Builds a pytorch hdf5 dataset
        Args:
            path (str): location of the hdf5 dataset
            task (str): string indicating label keys to return 
        """
        self.file_path = path
        self.dataset = None
        self.task = task
        self.batch_size = batch_size
        self.skip_negative_elev = skip_negative_elev
        # if self.skip_negative_elev:
        #     print("Skipping negative elevations")
        # else:
        #     print("Including negative elevations")

        if cue_type == 'voice_and_location':
            self.cue_key = 'voice_cue_target_loc'
        elif cue_type == 'voice':
            self.cue_key = "voice_cue_center_loc"
        elif cue_type == "location":
            self.cue_key = "loc_cue"

        if self.task == 'word':
            self.label_key = 'word_int'
        if self.task == 'location':
            self.label_key = ('label_loc_target_azim', 'label_loc_target_elev')
        if self.task == 'word_and_location':
            self.label_key = ('word_int', 'label_loc_target_azim', 'label_loc_target_elev')

        with h5py.File(self.file_path, 'r', swmr=True) as file:
            self.dataset_len = len(file['target']) // batch_size

    def azim_elev_to_label(self, azim, elev):
        if self.skip_negative_elev:
            return np.array(((elev / 10) * 72) + (azim / 5) + 1, dtype=np.int64)
        else:
            return np.array((((elev + 30) / 10) * 72) + (azim / 5), dtype=np.int64)

    def __getitem__(self, index):
        """
        Gets components of the hdf5 file that are used for training
        Args: 
            index (int): index into the hdf5 file
        Returns:
            [signal, target] : the training audio (signal) containing the preprocessing
              which may combine the foreground and background speech, and the target idx
              specified by target_keys. 
        """
        start = index * self.batch_size
        end = start + self.batch_size
        if self.dataset is None:
            self.dataset = h5py.File(self.file_path, 'r', swmr=True)

        cue = self.dataset[self.cue_key][start:end].transpose((0, 2, 1))
        if np.isnan(cue).all():
            cue[:] = 0
        foreground = self.dataset['target'][start:end].transpose((0, 2, 1))
        background = self.dataset['bg_scene'][start:end].transpose((0, 2, 1))

        # if self.skip_negative_elev and self.dataset['label_loc_target_elev'][start:end] < 0:
        #     return None, None, None, None

        if self.task == 'word':
            label = self.dataset[self.label_key][start:end]
        elif self.task == 'location':
            azim = self.dataset[self.label_key[0]][start:end]
            elev = self.dataset[self.label_key[1]][start:end]
            label = []
            for azim, elev in zip(azim, elev):
                label.append(self.azim_elev_to_label(azim, elev))
        else:
            word = self.dataset[self.label_key[0]][start:end]
            azim = self.dataset[self.label_key[1]][start:end]
            elev = self.dataset[self.label_key[2]][start:end]
            loc = []
            for azim, elev in zip(azim, elev):
                loc.append(self.azim_elev_to_label(azim, elev))
            label = []
            for word, loc in zip(word, loc):
                label = (word, loc)
        return cue, foreground, background, label

    def __len__(self):
        return self.dataset_len

In [20]:
dataset = BinauralAttentionDataset(**config['corpus'], batch_size=96, mode='train')
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False, num_workers=5, pin_memory=True)


for batch in tqdm(dataloader):
    break

1068 files in train concat dataset


  0%|          | 0/33108 [00:00<?, ?it/s]

In [12]:
cue, foreground, background, label = batch

In [13]:
label

tensor([[230, 103, 378, 596, 602, 580,  30, 714, 772,  20, 668, 444,  87,  70,
         632, 212,  98, 515, 364, 293, 557, 213, 381, 565, 427, 463, 755, 376,
         438, 471, 734, 332, 776, 437, 659, 239, 528, 419, 760, 340, 213, 745,
         391, 500, 692,  38, 219, 210, 152, 154, 329, 693, 481, 295, 201, 289,
         726, 568, 732, 592, 777, 261, 405, 120, 346, 392, 618, 431, 197, 516,
         261,   3, 253, 701, 703,  34, 155, 535, 713, 241, 746, 462, 380, 350,
         111, 764, 115, 620, 314, 294, 178, 551, 166, 491, 326, 784]],
       dtype=torch.int32)

In [14]:
cue_td = audio_transforms(cue.numpy().squeeze(), None)[0].squeeze()
scene = audio_transforms(foreground.numpy().squeeze(), background.numpy().squeeze())[0].squeeze()

## Test model i/0

In [15]:
module = module.train().cuda()
cue = cue_td[0].cuda()
scene = scene[0].cuda()

In [16]:
cue.shape, scene.shape, label[0,0].unsqueeze(0)

(torch.Size([2, 125000]),
 torch.Size([2, 125000]),
 tensor([230], dtype=torch.int32))

In [25]:
batch[0].shape

torch.Size([96, 2, 125000])

In [21]:
# check time for 100 batches with no I/0
# batch = (cue.unsqueeze(0), None, scene.unsqueeze(0), label[0,0].type(torch.LongTensor).unsqueeze(0).cuda())
counter = 0
limit = 200
for batch in tqdm(module.train_dataloader(), total=limit):
    batch = (batch[0].cuda(), None, batch[1].cuda(), batch[2].cuda())
    module.training_step(batch, 0)
    counter += 1
    if counter == limit:
        break
    

1068 files in train concat dataset
len training set = 3206467


  0%|          | 0/33401 [00:00<?, ?it/s]

IndexError: too many indices for tensor of dimension 2

In [ ]:
from IPython.display import Audio

In [ ]:
scene.shape

torch.Size([64, 2, 125000])

In [ ]:
Audio(scene[63], rate=50_000)